In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import optuna # 핵심 라이브러리
import joblib
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Models
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 1. 환경 설정
SEED = 42
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 [Step 1] Optuna 하이퍼파라미터 튜닝 시작 (Device: {device})")

In [ ]:
# 2. 데이터 로드
DATA_PATH = "./data/munch_labeled_RESUMED_200125.csv"
df = pd.read_csv(DATA_PATH)

# 라벨링 (4점 이상 Hard=1)
df['binary_label'] = df['score'].apply(lambda x: 1 if x >= 4.0 else 0)
df.dropna(subset=['original', 'binary_label'], inplace=True)

# 3. 임베딩 추출 (RoBERTa) - Feature Engineering
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

def get_embeddings(text_list, batch_size=64):
    embeddings = []
    print(f"🔄 임베딩 추출 중... (총 {len(text_list)}개)")
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch_texts = text_list[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
    return np.concatenate(embeddings, axis=0)

X_features = get_embeddings(df['original'].tolist())
y_labels = df['binary_label'].values

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_labels, test_size=0.2, random_state=SEED, stratify=y_labels
)

# ---------------------------------------------------------
# [핵심] Optuna를 이용한 베이지안 최적화
# ---------------------------------------------------------
N_TRIALS = 20  # 모델별 시도 횟수 (시간이 많으면 50~100 추천)

print(f"\n🔍 [Step 2] 하이퍼파라미터 탐색 시작 (모델당 {N_TRIALS}회 시도)")

# 1) XGBoost 최적화 함수
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': SEED,
        'n_jobs': -1,
        'verbosity': 0
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='macro')

# 2) LightGBM 최적화 함수
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'random_state': SEED,
        'n_jobs': -1,
        'verbose': -1
    }
    model = LGBMClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='macro')

# 3) CatBoost 최적화 함수
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0),
        'random_seed': SEED,
        'verbose': 0,
        'allow_writing_files': False
    }
    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return f1_score(y_test, preds, average='macro')

# 최적화 실행
print("  - 1/3 Tuning XGBoost...")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

print("  - 2/3 Tuning LightGBM...")
study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS)

print("  - 3/3 Tuning CatBoost...")
study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

# ---------------------------------------------------------
# [Step 3] 최적 파라미터로 최종 모델 생성 및 앙상블
# ---------------------------------------------------------
print("\n🏆 [Step 3] 최적 파라미터 적용 및 앙상블 학습")

best_xgb = XGBClassifier(**study_xgb.best_params, random_state=SEED, n_jobs=-1)
best_lgbm = LGBMClassifier(**study_lgbm.best_params, random_state=SEED, n_jobs=-1, verbose=-1)
best_cat = CatBoostClassifier(**study_cat.best_params, random_seed=SEED, verbose=0, allow_writing_files=False)

voting_clf = VotingClassifier(
    estimators=[
        ('XGB_Opt', best_xgb),
        ('LGBM_Opt', best_lgbm),
        ('CAT_Opt', best_cat)
    ],
    voting='soft'
)

# 앙상블 학습
voting_clf.fit(X_train, y_train)
final_pred = voting_clf.predict(X_test)

# 최종 평가
acc = accuracy_score(y_test, final_pred)
f1 = f1_score(y_test, final_pred, average='macro')

print("=" * 50)
print(f"🌟 최종 튜닝 모델 성능")
print(f"  - Accuracy: {acc:.4f}")
print(f"  - Macro F1: {f1:.4f}")
print("=" * 50)

print("\n📋 Classification Report:")
print(classification_report(y_test, final_pred, target_names=['Easy(0)', 'Hard(1)']))

# 결과 저장
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, final_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Pred: Easy', 'Pred: Hard'],
            yticklabels=['True: Easy', 'True: Hard'])
plt.title(f'Optuna Optimized Ensemble (Acc: {acc:.4f})')
plt.savefig('./output/optuna_ensemble_cm.png')

# 모델 파일 저장
joblib.dump(voting_clf, './output/smart_router_optuna_ensemble.pkl')
print(f"\n💾 최종 모델 저장 완료: ./output/smart_router_optuna_ensemble.pkl")